####Download and extract raw log

In [1]:
!wget 'https://zenodo.org/records/7119953/files/clue.zip?download=1' -O clue.zip

--2026-05-27 17:04:39--  https://zenodo.org/records/7119953/files/clue.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.184.98.114, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635105552 (606M) [application/octet-stream]
Saving to: ‘clue.zip’

clue.zip            100%[===================>] 605.68M  19.5MB/s    in 33s     

2026-05-27 17:05:12 (18.6 MB/s) - ‘clue.zip’ saved [635105552/635105552]



In [2]:
import os
print(os.listdir())
!unzip clue.zip

['.config', 'anomaly_scores_global_baseline_inconsistent.csv', 'isolation_forest_scores.csv', 'anomaly_scores_global_baseline.csv', 'clue.zip', 'isolation_forest_scores_inconsistent.csv', 'sample_data']
Archive:  clue.zip
  inflating: clue.json               


In [3]:
!ls -lh clue.json

-rw-r--r-- 1 root root 14G Oct  1  2022 clue.json


####Manageable Dataset 1GB

In [4]:
import json

target_bytes = 1_073_741_824  # 1GB
record_size = 297
max_records = target_bytes // record_size

print(f"Sampling {max_records:,} records...")

in_path = 'clue.json'
out_path = 'clue_sample_1GB.jsonl'

count = 0
total_bytes = 0

with open(in_path, 'r') as fin, open(out_path, 'w') as fout:
    for line in fin:
        if count >= max_records or total_bytes > target_bytes:
            break
        fout.write(line)
        count += 1
        total_bytes += len(line.encode('utf-8'))

        if count % 500000 == 0:
            print(f"  Wrote {count:,} records ({total_bytes/1024/1024:.1f} MB)")

print(f"\nDone. Wrote {count:,} records ({total_bytes/1024/1024:.1f} MB)")
print(f"File: {out_path}")

Sampling 3,615,292 records...
  Wrote 500,000 records (117.6 MB)
  Wrote 1,000,000 records (297.7 MB)
  Wrote 1,500,000 records (468.6 MB)
  Wrote 2,000,000 records (603.4 MB)
  Wrote 2,500,000 records (752.1 MB)
  Wrote 3,000,000 records (889.7 MB)

Done. Wrote 3,479,163 records (1024.0 MB)
File: clue_sample_1GB.jsonl


In [5]:
!ls -lh clue_sample_1GB.jsonl

-rw-r--r-- 1 root root 1.1G May 27 17:07 clue_sample_1GB.jsonl


In [6]:
with open('clue_sample_1GB.jsonl', 'r') as f:
    for i in range(3):
        print(f.readline().strip())

{"params": {"path": "/useful-turquoise-cardinal-historian"}, "type": "file_accessed", "time": "2017-07-07T08:57:57Z", "uid": "yellow-chocolate-carp-busdriver", "id": 1, "uidType": "name"}
{"params": {"path": "/useful-turquoise-cardinal-historian", "run": true}, "type": "file_updated", "time": "2017-07-07T08:58:02Z", "uid": "yellow-chocolate-carp-busdriver", "id": 2, "uidType": "name"}
{"params": {"path": "/useful-turquoise-cardinal-historian", "run": true}, "type": "file_written", "time": "2017-07-07T08:58:02Z", "uid": "yellow-chocolate-carp-busdriver", "id": 3, "uidType": "name"}


####Basic exploratory scan: users, roles, directories, hours

In [7]:
import json
from collections import defaultdict, Counter

# Track users and their roles
user_roles = {}  # uid -> role
user_event_count = Counter()
user_dirs = defaultdict(set)  # uid -> set of top-level directories
user_hours = defaultdict(list)  # uid -> list of hours

print("Scanning file...")

with open('clue_sample_1GB.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i % 500000 == 0:
            print(f"  Processed {i:,} lines...")

        try:
            data = json.loads(line)
            uid = data.get('uid')
            role = data.get('role')
            time_str = data.get('time', '')
            params = data.get('params', {})
            path = params.get('path', '') if isinstance(params, dict) else ''

            if not uid:
                continue

            # Track role (first seen)
            if role and uid not in user_roles:
                user_roles[uid] = role

            # Count events per user
            user_event_count[uid] += 1

            # Track hour
            if time_str:
                hour = int(time_str[11:13]) if len(time_str) > 11 else 0
                user_hours[uid].append(hour)

            # Track top-level directory
            if path and path.startswith('/'):
                parts = path.strip('/').split('/')
                if parts:
                    user_dirs[uid].add(parts[0])

        except:
            continue

print(f"\nScan complete.")
print(f"Total users: {len(user_event_count)}")
print(f"Users with role field: {len(user_roles)}")

Scanning file...
  Processed 0 lines...
  Processed 500,000 lines...
  Processed 1,000,000 lines...
  Processed 1,500,000 lines...
  Processed 2,000,000 lines...
  Processed 2,500,000 lines...
  Processed 3,000,000 lines...

Scan complete.
Total users: 477
Users with role field: 15


#####all 15 role-labeled users have the role on 100% of their events
none of them have multiple roles

In [8]:
# Check role consistency per user
user_role_consistency = defaultdict(lambda: {'role_seen': None, 'with_role': 0, 'without_role': 0})

print("Checking role consistency...")

with open('clue_sample_1GB.jsonl', 'r') as f:
    for line in f:
        try:
            data = json.loads(line)
            uid = data.get('uid')
            role = data.get('role')

            if not uid:
                continue

            if role:
                user_role_consistency[uid]['with_role'] += 1
                if user_role_consistency[uid]['role_seen'] is None:
                    user_role_consistency[uid]['role_seen'] = role
                elif user_role_consistency[uid]['role_seen'] != role:
                    user_role_consistency[uid]['role_seen'] = 'MULTIPLE'
            else:
                user_role_consistency[uid]['without_role'] += 1

        except:
            continue

print("\n=== RESULTS ===\n")

# Question 1: Do users with role have it on ALL events?
users_with_role = {uid: data for uid, data in user_role_consistency.items()
                   if data['with_role'] > 0}

perfect_users = 0
for uid, data in users_with_role.items():
    total = data['with_role'] + data['without_role']
    if data['without_role'] == 0:
        perfect_users += 1
        print(f"{uid}: role='{data['role_seen']}' on ALL {data['with_role']} events")
    else:
        pct = (data['with_role'] / total) * 100
        print(f" {uid}: role='{data['role_seen']}' on {data['with_role']}/{total} events ({pct:.1f}%)")

print(f"\nUsers with role always present: {perfect_users}/{len(users_with_role)} ({perfect_users/len(users_with_role)*100:.1f}%)")

# Question 2: Can a user have more than one role?
users_with_multiple = [uid for uid, data in users_with_role.items()
                       if data['role_seen'] == 'MULTIPLE']

print(f"\nUsers with multiple different roles: {len(users_with_multiple)}")

if len(users_with_multiple) == 0:
    print(" Each user has a SINGLE, CONSISTENT role.")
else:
    print(f" Some users have multiple roles: {users_with_multiple}")

Checking role consistency...

=== RESULTS ===

central-silver-slug-zookeeper: role='technical' on ALL 7914 events
little-amber-minnow-reporter: role='external' on ALL 108103 events
monthly-emerald-bug-zoologist: role='technical' on ALL 3907 events
spectacular-copper-cheetah-postman: role='technical' on ALL 7473 events
inevitable-olive-meerkat-metallurgist: role='administration' on ALL 2968 events
stupid-orange-ant-tacker: role='management' on ALL 803606 events
ethnic-lavender-gerbil-gamingclubmanager: role='administration' on ALL 5593 events
lively-pink-narwhal-yachtmaster: role='technical' on ALL 2223 events
fast-coffee-ocelot-arbitrator: role='technical' on ALL 7168 events
communist-indigo-possum-forester: role='sales' on ALL 1508 events
varying-brown-nightingale-book-keeper: role='technical' on ALL 278 events
accessible-coral-ferret-repairer: role='technical' on ALL 184 events
selected-beige-vole-recorder: role='administration' on ALL 2190 events
northern-magenta-limpet-preacher: ro

####Next, it builds “behavioral features” but only for events that include a role (so, only those 15 users).

In [9]:
import pandas as pd
import numpy as np
from collections import defaultdict

# Re-scan to collect behavioral features per user
user_features = defaultdict(lambda: {
    'role': None,
    'total_events': 0,
    'unique_dirs': set(),
    'night_events': 0,
    'total_hours': set(),
    'file_events': 0,
    'login_events': 0
})

print("Collecting behavioral data for role users...")

with open('clue_sample_1GB.jsonl', 'r') as f:
    for line in f:
        try:
            data = json.loads(line)
            uid = data.get('uid')
            role = data.get('role')

            if not uid or not role:
                continue

            params = data.get('params', {})
            path = params.get('path', '') if isinstance(params, dict) else ''
            event_type = data.get('type', '')
            time_str = data.get('time', '')

            feat = user_features[uid]
            feat['role'] = role
            feat['total_events'] += 1

            if path and path.startswith('/'):
                parts = path.strip('/').split('/')
                if parts:
                    feat['unique_dirs'].add(parts[0])

            if time_str and len(time_str) > 11:
                hour = int(time_str[11:13])
                feat['total_hours'].add(hour)
                if hour <= 5:
                    feat['night_events'] += 1

            if event_type == 'file_accessed':
                feat['file_events'] += 1
            elif event_type in ['login_attempt', 'login_successful']:
                feat['login_events'] += 1

        except:
            continue

# Build comparison table
print("\n=== BEHAVIOR BY ROLE ===\n")

role_stats = defaultdict(lambda: {
    'users': [],
    'events_per_day': [],  # approximate (total/89)
    'unique_dirs': [],
    'night_frac': [],
    'active_hours': [],
    'file_ratio': []
})

for uid, feat in user_features.items():
    role = feat['role']
    days_active = len(feat['total_hours'])  # rough proxy
    events_per_day = feat['total_events'] / max(89, days_active)
    night_frac = feat['night_events'] / max(1, feat['total_events'])
    active_hours = len(feat['total_hours'])
    file_ratio = feat['file_events'] / max(1, feat['total_events'])

    role_stats[role]['users'].append(uid)
    role_stats[role]['events_per_day'].append(events_per_day)
    role_stats[role]['unique_dirs'].append(len(feat['unique_dirs']))
    role_stats[role]['night_frac'].append(night_frac)
    role_stats[role]['active_hours'].append(active_hours)
    role_stats[role]['file_ratio'].append(file_ratio)

# Print summary
for role, stats in role_stats.items():
    n = len(stats['users'])
    print(f"\n{'='*50}")
    print(f"ROLE: {role.upper()} ({n} users)")
    print(f"{'='*50}")

    print(f"\n Unique directories accessed:")
    dirs = stats['unique_dirs']
    print(f"   Min: {min(dirs)} | Max: {max(dirs)} | Mean: {np.mean(dirs):.1f} | Std: {np.std(dirs):.1f}")

    print(f"\n Events per day (approx):")
    epd = stats['events_per_day']
    print(f"   Min: {min(epd):.0f} | Max: {max(epd):.0f} | Mean: {np.mean(epd):.0f} | Std: {np.std(epd):.0f}")

    print(f"\n Night fraction (0-5 AM):")
    nf = stats['night_frac']
    print(f"   Min: {min(nf):.2%} | Max: {max(nf):.2%} | Mean: {np.mean(nf):.2%} | Std: {np.std(nf):.2%}")

    print(f"\n Active hours per day:")
    ah = stats['active_hours']
    print(f"   Min: {min(ah)} | Max: {max(ah)} | Mean: {np.mean(ah):.1f} | Std: {np.std(ah):.1f}")

    print(f"\n File access ratio (file_events / total_events):")
    fr = stats['file_ratio']
    print(f"   Min: {min(fr):.2%} | Max: {max(fr):.2%} | Mean: {np.mean(fr):.2%} | Std: {np.std(fr):.2%}")


=== BEHAVIOR BY ROLE ===


ROLE: TECHNICAL (8 users)

 Unique directories accessed:
   Min: 3 | Max: 825 | Mean: 117.8 | Std: 267.6

 Events per day (approx):
   Min: 2 | Max: 89 | Mean: 46 | Std: 33

 Night fraction (0-5 AM):
   Min: 0.42% | Max: 26.44% | Mean: 7.92% | Std: 7.73%

 Active hours per day:
   Min: 18 | Max: 24 | Mean: 22.6 | Std: 2.4

 File access ratio (file_events / total_events):
   Min: 14.39% | Max: 99.37% | Mean: 45.24% | Std: 28.35%

ROLE: EXTERNAL (1 users)

 Unique directories accessed:
   Min: 11 | Max: 11 | Mean: 11.0 | Std: 0.0

 Events per day (approx):
   Min: 1215 | Max: 1215 | Mean: 1215 | Std: 0

 Night fraction (0-5 AM):
   Min: 15.27% | Max: 15.27% | Mean: 15.27% | Std: 0.00%

 Active hours per day:
   Min: 24 | Max: 24 | Mean: 24.0 | Std: 0.0

 File access ratio (file_events / total_events):
   Min: 95.67% | Max: 95.67% | Mean: 95.67% | Std: 0.00%

ROLE: ADMINISTRATION (3 users)

 Unique directories accessed:
   Min: 2 | Max: 7 | Mean: 5.0 | Std: 2.2

Generating csv files

In [10]:
import json
import pandas as pd
from collections import defaultdict
from datetime import datetime

# Specify your target users here
users = [
    "spectacular-copper-cheetah-postman",
    "ready-silver-angelfish-quarryworker"
]
# You can extend to more users by adding their uids above

# Dictionary for per-user, per-day metrics
features = {
    user: defaultdict(lambda: {
        "events_total": 0,
        "hours": set(),                  # set of hours active
        "file_events": 0,
        "login_attempt": 0,
        "login_successful": 0,
        "unique_paths": set(),
        "file_accessed": 0,
        "path_counts": defaultdict(int)  # path -> count
    })
    for user in users
}

# Parse the log
with open("clue_sample_1GB.jsonl", "r") as f:
    for line in f:
        try:
            data = json.loads(line)
            uid = data.get("uid")
            if uid not in users:
                continue
            time_str = data.get("time", "")
            if not time_str:
                continue
            dt = datetime.fromisoformat(time_str.replace("Z", "+00:00"))
            date = dt.date()
            hour = dt.hour
            typ = data.get("type")
            params = data.get("params", {})
            path = params.get("path", "") if isinstance(params, dict) else ""

            feats = features[uid][date]
            feats["events_total"] += 1
            feats["hours"].add(hour)
            if typ == "file_accessed":
                feats["file_accessed"] += 1
            if typ.startswith("file_"):
                feats["file_events"] += 1
                if path:
                    feats["unique_paths"].add(path)
                    feats["path_counts"][path] += 1
            if typ == "login_attempt":
                feats["login_attempt"] += 1
            if typ == "login_successful":
                feats["login_successful"] += 1
        except Exception:
            continue

# Functions for engineered feature columns
def compute_path_reuse_ratio(path_counts):
    total = sum(path_counts.values())
    if total == 0:
        return 1.0  # if no paths accessed, treat as maximum reuse
    unique = len(path_counts)
    return unique / total

def compute_login_success_rate(logins, attempts):
    return logins / attempts if attempts > 0 else 1.0

# Write per-user daily features as CSV
for uid in users:
    rows = []
    for date, d in sorted(features[uid].items()):
        row = {
            "date": date,
            "events_total": d["events_total"],
            "active_hours": len(d["hours"]),
            "file_events": d["file_events"],
            "file_accessed": d["file_accessed"],
            "unique_paths": len(d["unique_paths"]),
            "login_successful": d["login_successful"],
            "login_attempt": d["login_attempt"],
            "path_reuse_ratio": compute_path_reuse_ratio(d["path_counts"]),
            "login_success_rate": compute_login_success_rate(d["login_successful"], d["login_attempt"])
        }
        rows.append(row)
    df = pd.DataFrame(rows)
    out_name = f"daily_features_{uid}.csv"
    df.to_csv(out_name, index=False)
    print(f"Saved: {out_name} ({len(df)} days)")

Saved: daily_features_spectacular-copper-cheetah-postman.csv (88 days)
Saved: daily_features_ready-silver-angelfish-quarryworker.csv (83 days)


####Pick two users and list their event types (one consistent and other inconsistent)

In [11]:
from collections import Counter

users = [
    "spectacular-copper-cheetah-postman",
    "ready-silver-angelfish-quarryworker"
]

event_types = Counter()

with open("clue_sample_1GB.jsonl", "r") as f:
    for line in f:
        try:
            data = json.loads(line)
            uid = data.get("uid")
            if uid in users:
                event_types[data.get("type")] += 1
        except:
            continue

print("Event types found for both users:\n")
for event, count in event_types.most_common():
    print(f"  {event}: {count:,}")

Event types found for both users:

  file_accessed: 15,631
  login_attempt: 3,639
  login_successful: 3,530
  file_written: 363
  file_created: 307
  file_deleted: 191
  deleted_from_trashbin: 95
  file_updated: 56
  file_renamed: 23
  shared_user: 2
  logout_occured: 1


# Qualitative Study of flagged events:

### Spike Mapping for both users

In [12]:
import pandas as pd
import json
from collections import Counter

# ===================================================
# 1. USER CONFIGS
# ===================================================
user_configs = [
    {
        "uid": "spectacular-copper-cheetah-postman",
        "label": "Consistent",
        "base_csv": "anomaly_scores_global_baseline.csv",
        "if_csv": "isolation_forest_scores.csv",
        "features_csv": "daily_features_spectacular-copper-cheetah-postman.csv",
    },
    {
        "uid": "ready-silver-angelfish-quarryworker",
        "label": "Inconsistent",
        "base_csv": "anomaly_scores_global_baseline_inconsistent.csv",
        "if_csv": "isolation_forest_scores_inconsistent.csv",
        "features_csv": "daily_features_ready-silver-angelfish-quarryworker.csv",
    },
]

# ===================================================
# 2. SHARED HELPERS
# ===================================================
def get_hourly_distribution(uid, date):
    """Get event counts per hour from raw logs."""
    hourly = Counter()
    with open("clue_sample_1GB.jsonl", "r") as f:
        for line in f:
            try:
                data = json.loads(line)
                if data.get("uid") == uid and data.get("time", "").startswith(date):
                    hour = int(data.get("time", "")[11:13])
                    hourly[hour] += 1
            except:
                continue
    return hourly

def classify_attack(top_contributors):
    if pd.isna(top_contributors) or top_contributors == "No features exceeded p95":
        return 'NONE', []
    spikes = [s.split('=')[0] for s in top_contributors.split('; ') if '=' in s]
    if 'file_events' in spikes and 'unique_paths' in spikes:
        return 'DATA_THEFT', spikes
    if 'events_total' in spikes and 'active_hours' in spikes:
        return 'BOT_OR_MASS_ACTIVITY', spikes
    if 'unique_dir1' in spikes or 'unique_dir2' in spikes:
        return 'DIRECTORY_TRAVERSAL', spikes
    if 'login_attempt' in spikes:
        return 'LOGIN_ACTIVITY', spikes
    if 'night_fraction' in spikes:
        return 'OFF_HOURS', spikes
    if 'unique_types' in spikes:
        return 'DIVERSE_ACTIVITY', spikes
    if 'events_total' in spikes and len(spikes) == 1:
        return 'MASS_ACTIVITY', spikes
    if 'path_reuse_ratio' in spikes and len(spikes) == 1:
        return 'PATH_REUSE_ANOMALY', spikes
    return 'UNKNOWN', spikes

# ===================================================
# 3. VERIFICATION HELPERS
# ===================================================
def verify_data_theft(date, attack_type, df_features):
    if attack_type != 'DATA_THEFT':
        return None
    row = df_features[df_features['date'] == pd.to_datetime(date).date()]
    if row.empty:
        return 'NO_DATA'
    current = row['path_reuse_ratio'].iloc[0]
    historical = df_features[df_features['date'] < pd.to_datetime(date).date()]
    if len(historical) == 0:
        return 'INSUFFICIENT_HISTORY'
    return 'CONFIRMED' if current < historical['path_reuse_ratio'].median() else 'SUSPICIOUS'

def verify_login(date, attack_type, df_features):
    if attack_type != 'LOGIN_ACTIVITY':
        return None
    row = df_features[df_features['date'] == pd.to_datetime(date).date()]
    if row.empty:
        return 'NO_DATA'
    rate = row['login_success_rate'].iloc[0]
    if rate < 0.5:
        return 'BRUTE_FORCE'
    elif rate > 0.8:
        return 'FALSE_POSITIVE_BUSY_DAY'
    return 'SUSPICIOUS'

def verify_bot(date, attack_type, uid, df_features):
    if attack_type != 'BOT_OR_MASS_ACTIVITY':
        return None
    row = df_features[df_features['date'] == pd.to_datetime(date).date()]
    if row.empty:
        return 'NO_DATA'
    events = row['events_total'].iloc[0]
    hours = row['active_hours'].iloc[0]
    hourly = get_hourly_distribution(uid, date)
    if not hourly:
        return 'NO_HOURLY_DATA'
    peak_concentration = max(hourly.values()) / events
    return 'BOT_ACTIVITY' if peak_concentration > 0.5 and hours < 4 else 'MASS_ACTIVITY'
def verify_path_reuse(date, attack_type, df_features):
    if attack_type != 'PATH_REUSE_ANOMALY':
        return None
    row = df_features[df_features['date'] == pd.to_datetime(date).date()]
    if row.empty:
        return 'NO_DATA'
    current = row['path_reuse_ratio'].iloc[0]
    historical = df_features[df_features['date'] < pd.to_datetime(date).date()]
    if len(historical) == 0:
        return 'INSUFFICIENT_HISTORY'
    return 'CONFIRMED' if current < historical['path_reuse_ratio'].median() else 'SUSPICIOUS'

# ===================================================
# 4. MAIN LOOP
# ===================================================
from IPython.display import HTML, display

for config in user_configs:
    uid = config["uid"]
    label = config["label"]

    df_base = pd.read_csv(config["base_csv"])
    df_if = pd.read_csv(config["if_csv"])
    df_features = pd.read_csv(config["features_csv"])
    df_features['date'] = pd.to_datetime(df_features['date']).dt.date

    df_base = df_base[df_base['anomaly_score'] > 0]

    if_lookup = dict(zip(df_if['date'], df_if['iso_pred'] == -1))
    df_base['if_flagged'] = df_base['date'].map(if_lookup).fillna(False)
    df_base['confidence'] = df_base['if_flagged'].apply(lambda x: 'HIGH' if x else 'MEDIUM')

    classification = df_base['top_contributors'].apply(classify_attack)
    df_base['attack_type'] = classification.apply(lambda x: x[0])
    df_base['spikes'] = classification.apply(lambda x: x[1])

    df_base['data_theft_verification'] = df_base.apply(
        lambda r: verify_data_theft(r['date'], r['attack_type'], df_features), axis=1)
    df_base['login_verification'] = df_base.apply(
        lambda r: verify_login(r['date'], r['attack_type'], df_features), axis=1)
    df_base['bot_verification'] = df_base.apply(
        lambda r: verify_bot(r['date'], r['attack_type'], uid, df_features), axis=1)
    df_base['path_reuse_verification'] = df_base.apply(
        lambda r: verify_path_reuse(r['date'], r['attack_type'], df_features), axis=1)

    def format_verification(row):
        parts = []
        if row['data_theft_verification']:
            parts.append(f"DATA_THEFT: {row['data_theft_verification']}")
        if row['login_verification']:
            parts.append(f"LOGIN: {row['login_verification']}")
        if row['bot_verification']:
            parts.append(f"BOT/MASS: {row['bot_verification']}")
        if row['attack_type'] == 'UNKNOWN' and row['spikes']:
            parts.append(f"⚠️ Spikes: {', '.join(row['spikes'])}")
        if row['path_reuse_verification']:
            parts.append(f"PATH_REUSE: {row['path_reuse_verification']}")
        return "<br>".join(parts) if parts else "—"

    def confidence_badge(conf):
        color = "#6a0dad" if conf == "HIGH" else "#f0a500"
        return f'<span style="background:{color};color:white;padding:2px 8px;border-radius:4px;font-size:0.85em;">{conf}</span>'

    def attack_color(attack):
        colors = {
            'DATA_THEFT': '#c0392b',
            'BOT_OR_MASS_ACTIVITY': '#e67e22',
            'BOT_ACTIVITY': '#e67e22',
            'MASS_ACTIVITY': '#e67e22',
            'LOGIN_ACTIVITY': '#2980b9',
            'BRUTE_FORCE': '#c0392b',
            'DIRECTORY_TRAVERSAL': '#8e44ad',
            'OFF_HOURS': '#16a085',
            'UNKNOWN': '#7f8c8d',
            'NONE': '#bdc3c7',
        }
        color = colors.get(attack, '#333')
        return f'<span style="color:{color};font-weight:bold;">{attack}</span>'

    rows_html = ""
    for _, row in df_base.iterrows():
        rows_html += f"""
        <tr>
            <td>{row['date']}</td>
            <td>{attack_color(row['attack_type'])}</td>
            <td>{confidence_badge(row['confidence'])}</td>
            <td>{'✅' if row['if_flagged'] else '—'}</td>
            <td style="font-size:0.9em;">{format_verification(row)}</td>
        </tr>"""

    html = f"""
    <h3 style="font-family:Arial,sans-serif;color:#6a0dad;">
        Flagged Days — {label} User ({uid})
    </h3>
    <table style="border-collapse:collapse;width:100%;font-family:Arial,sans-serif;">
        <thead>
            <tr style="background:#6a0dad;color:white;">
                <th style="padding:8px;">Date</th>
                <th style="padding:8px;">Attack Type</th>
                <th style="padding:8px;">Confidence</th>
                <th style="padding:8px;">IF Flagged</th>
                <th style="padding:8px;">Verification</th>
            </tr>
        </thead>
        <tbody>
            {rows_html}
        </tbody>
    </table>
    <br>
    """
    display(HTML(html))

Date,Attack Type,Confidence,IF Flagged,Verification
2017-07-17,DATA_THEFT,HIGH,✅,DATA_THEFT: SUSPICIOUS
2017-07-28,DATA_THEFT,HIGH,✅,DATA_THEFT: SUSPICIOUS
2017-08-24,DATA_THEFT,MEDIUM,—,DATA_THEFT: CONFIRMED
2017-08-01,DIRECTORY_TRAVERSAL,MEDIUM,—,—
2017-07-15,BOT_OR_MASS_ACTIVITY,HIGH,✅,BOT/MASS: MASS_ACTIVITY
2017-09-18,OFF_HOURS,HIGH,✅,—
2017-07-08,LOGIN_ACTIVITY,MEDIUM,—,LOGIN: FALSE_POSITIVE_BUSY_DAY
2017-08-08,DIVERSE_ACTIVITY,MEDIUM,—,—
2017-07-20,UNKNOWN,MEDIUM,—,"⚠️ Spikes: file_events, path_depth_mean"
2017-10-02,PATH_REUSE_ANOMALY,MEDIUM,—,PATH_REUSE: CONFIRMED


Date,Attack Type,Confidence,IF Flagged,Verification
2017-07-11,DATA_THEFT,HIGH,✅,DATA_THEFT: SUSPICIOUS
2017-08-18,DIRECTORY_TRAVERSAL,HIGH,✅,—
2017-07-23,DIRECTORY_TRAVERSAL,HIGH,✅,—
2017-08-24,DATA_THEFT,MEDIUM,—,DATA_THEFT: CONFIRMED
2017-07-17,DIRECTORY_TRAVERSAL,HIGH,✅,—
2017-08-30,UNKNOWN,HIGH,✅,"⚠️ Spikes: path_reuse_ratio, active_hours"
2017-07-14,DIRECTORY_TRAVERSAL,MEDIUM,—,—
2017-07-21,DIRECTORY_TRAVERSAL,MEDIUM,—,—
2017-09-13,LOGIN_ACTIVITY,MEDIUM,—,LOGIN: FALSE_POSITIVE_BUSY_DAY
2017-10-02,OFF_HOURS,MEDIUM,—,—


#### recapulative table of detection algorithm

In [13]:
from IPython.display import HTML, display

html_table = """
<style>
.th-attack { background-color: #6a0dad; color: white; }
.th-spike { background-color: #7b1fa2; color: white; }
.th-verif { background-color: #8e24aa; color: white; }
.th-seuil { background-color: #9c27b0; color: white; }
.th-decision { background-color: #ab47bc; color: white; }
.th-conf { background-color: #ba68c8; color: white; }
td { padding: 8px; border: 1px solid #ddd; }
tr:hover { background-color: #f5f0ff; }
.not-impl { color: #aaa; font-style: italic; }
</style>

<table style="border-collapse: collapse; width: 100%; font-family: Arial, sans-serif;">
<thead>
<tr style="background-color: #6a0dad; color: white;">
<th class="th-attack">Attaque</th>
<th class="th-spike">Spike(s) requis</th>
<th class="th-verif">Vérification</th>
<th class="th-seuil">Seuil</th>
<th class="th-decision">Décision</th>
<th class="th-conf">Confiance</th>
</tr>
</thead>
<tbody>
<tr>
  <td>Vol de données</td>
  <td>file_events + unique_paths</td>
  <td>path_reuse_ratio</td>
  <td>&lt; médiane utilisateur</td>
  <td style="color: #6a0dad; font-weight: bold;">DATA_THEFT</td>
  <td>HIGH si IF flagged, sinon MEDIUM</td>
</tr>
<tr>
  <td>Réutilisation de chemins</td>
  <td>path_reuse_ratio seul</td>
  <td>path_reuse_ratio</td>
  <td>&lt; médiane utilisateur</td>
  <td style="color: #6a0dad; font-weight: bold;">PATH_REUSE_ANOMALY</td>
  <td>HIGH si IF flagged, sinon MEDIUM</td>
</tr>
<tr>
  <td>Bot</td>
  <td>events_total + active_hours</td>
  <td>Pic horaire &gt; 50% ET active_hours &lt; 4</td>
  <td>&gt; 50% en 1h et &lt; 4h actives</td>
  <td style="color: #6a0dad; font-weight: bold;">BOT_ACTIVITY</td>
  <td>HIGH si IF flagged, sinon MEDIUM</td>
</tr>
<tr>
  <td>Activité massive</td>
  <td>events_total + active_hours</td>
  <td>Pic horaire ≤ 50% OU active_hours ≥ 4</td>
  <td>-</td>
  <td>MASS_ACTIVITY</td>
  <td>HIGH si IF flagged, sinon MEDIUM</td>
</tr>
<tr>
  <td>Brute force</td>
  <td>login_attempt</td>
  <td>login_success_rate</td>
  <td>&lt; 50%</td>
  <td style="color: #6a0dad; font-weight: bold;">BRUTE_FORCE</td>
  <td>HIGH si IF flagged, sinon MEDIUM</td>
</tr>
<tr>
  <td>Faux positif login</td>
  <td>login_attempt</td>
  <td>login_success_rate</td>
  <td>&gt; 80%</td>
  <td style="color: #888; font-style: italic;">FALSE_POSITIVE_BUSY_DAY</td>
  <td>-</td>
</tr>
<tr>
  <td>Traversal</td>
  <td>unique_dir1 ou unique_dir2</td>
  <td>-</td>
  <td>-</td>
  <td>DIRECTORY_TRAVERSAL</td>
  <td>HIGH si IF flagged, sinon MEDIUM</td>
</tr>
<tr>
  <td>Horaires décalés</td>
  <td>night_fraction</td>
  <td>-</td>
  <td>-</td>
  <td>OFF_HOURS</td>
  <td>HIGH si IF flagged, sinon MEDIUM</td>
</tr>
<tr>
  <td>Activité diversifiée</td>
  <td>unique_types</td>
  <td>-</td>
  <td>-</td>
  <td>DIVERSE_ACTIVITY</td>
  <td>HIGH si IF flagged, sinon MEDIUM</td>
</tr>
<tr>
  <td>Exfiltration via partage</td>
  <td>shared_user</td>
  <td>-</td>
  <td>&gt;= 1</td>
  <td class="not-impl">Non implémenté</td>
  <td>-</td>
</tr>
<tr>
  <td>Non classifié</td>
  <td>Aucune combinaison reconnue</td>
  <td>-</td>
  <td>-</td>
  <td>UNKNOWN (spikes listés)</td>
  <td>HIGH si IF flagged, sinon MEDIUM</td>
</tr>
</tbody>
</table>
"""

display(HTML(html_table))

Attaque,Spike(s) requis,Vérification,Seuil,Décision,Confiance
Vol de données,file_events + unique_paths,path_reuse_ratio,< médiane utilisateur,DATA_THEFT,"HIGH si IF flagged, sinon MEDIUM"
Réutilisation de chemins,path_reuse_ratio seul,path_reuse_ratio,< médiane utilisateur,PATH_REUSE_ANOMALY,"HIGH si IF flagged, sinon MEDIUM"
Bot,events_total + active_hours,Pic horaire > 50% ET active_hours < 4,> 50% en 1h et < 4h actives,BOT_ACTIVITY,"HIGH si IF flagged, sinon MEDIUM"
Activité massive,events_total + active_hours,Pic horaire ≤ 50% OU active_hours ≥ 4,-,MASS_ACTIVITY,"HIGH si IF flagged, sinon MEDIUM"
Brute force,login_attempt,login_success_rate,< 50%,BRUTE_FORCE,"HIGH si IF flagged, sinon MEDIUM"
Faux positif login,login_attempt,login_success_rate,> 80%,FALSE_POSITIVE_BUSY_DAY,-
Traversal,unique_dir1 ou unique_dir2,-,-,DIRECTORY_TRAVERSAL,"HIGH si IF flagged, sinon MEDIUM"
Horaires décalés,night_fraction,-,-,OFF_HOURS,"HIGH si IF flagged, sinon MEDIUM"
Activité diversifiée,unique_types,-,-,DIVERSE_ACTIVITY,"HIGH si IF flagged, sinon MEDIUM"
Exfiltration via partage,shared_user,-,>= 1,Non implémenté,-


#Qualitative study of rare events

In [14]:
from IPython.display import HTML, display

html_table = """
<style>
.th-attack { background-color: #6a0dad; color: white; }
.th-event { background-color: #7b1fa2; color: white; }
.th-window { background-color: #8e24aa; color: white; }
.th-stable { background-color: #9c27b0; color: white; }
.th-instable { background-color: #d95b5b; color: white; }
td { padding: 10px; border: 1px solid #ddd; }
tr:hover { background-color: #f5f0ff; }
.note { font-size: 0.9em; margin-top: 15px; color: #555; }
</style>

<p style="font-size: 1.1em; margin-bottom: 10px;"><strong>Phase 2 : Detection d'evenements rares et sequences</strong><br>
Seuil = (max historique par fenetre) × multiplicateur.<br>
Multiplicateur plus eleve pour l'utilisateur instable (variabilite naturelle plus grande).</p>

<table style="border-collapse: collapse; width: 100%; font-family: Arial, sans-serif;">
<thead>
<tr style="background-color: #6a0dad; color: white;">
<th class="th-attack">Attaque</th>
<th class="th-event">Evenement(s)</th>
<th class="th-window">Fenetre</th>
<th class="th-stable">Seuil (stable)</th>
<th class="th-instable">Seuil (instable)</th>
</tr>
</thead>
<tbody>
<tr style="background-color: #f9f0ff;">
<td style="font-weight: bold;">Exfiltration via partage</td>
<td><code>shared_user</code></td>
<td>instant</td>
<td>>= 1</td>
<td style="background-color: #ffe0e0;">>= 1</td>
</tr>
<tr>
<td style="font-weight: bold;">Ransomware</td>
<td><code>file_written</code></td>
<td>1 minute</td>
<td>> max × 2</td>
<td style="background-color: #ffe0e0;">>> max × 3</td>
</tr>
<tr style="background-color: #f9f0ff;">
<td style="font-weight: bold;">Suppression massive</td>
<td><code>file_deleted</code></td>
<td>5 minutes</td>
<td>> max × 2</td>
<td style="background-color: #ffe0e0;">>> max × 3</td>
</tr>
<tr>
<td style="font-weight: bold;">Depot malveillant</td>
<td><code>file_created</code></td>
<td>5 minutes</td>
<td>> max × 2</td>
<td style="background-color: #ffe0e0;">>> max × 3</td>
</tr>
<tr style="background-color: #f9f0ff;">
<td style="font-weight: bold;">Compte vole</td>
<td><code>login_successful</code> → <code>file_accessed</code></td>
<td>1 min apres login</td>
<td>> max × 2</td>
<td style="background-color: #ffe0e0;">>> max × 3</td>
</tr>
</tbody>
</table>

<p class="note"><strong>Note :</strong> Les seuils sont calcules a partir du maximum historique de chaque utilisateur. L'utilisateur instable utilise un multiplicateur plus eleve (×3) pour eviter les faux positifs lies a sa variabilite naturelle.</p>
"""

display(HTML(html_table))

Attaque,Evenement(s),Fenetre,Seuil (stable),Seuil (instable)
Exfiltration via partage,shared_user,instant,>= 1,>= 1
Ransomware,file_written,1 minute,> max × 2,>> max × 3
Suppression massive,file_deleted,5 minutes,> max × 2,>> max × 3
Depot malveillant,file_created,5 minutes,> max × 2,>> max × 3
Compte vole,login_successful → file_accessed,1 min apres login,> max × 2,>> max × 3


In [15]:
import json
from collections import deque
from datetime import datetime, timedelta

def sliding_window_max(uid, event_type, window_seconds, log_path="clue_sample_1GB.jsonl"):
    """
    Computes the maximum number of 'event_type' events by 'uid' in any
    'window_seconds' sliding window.
    """
    timestamps = []
    with open(log_path, "r") as f:
        for line in f:
            try:
                data = json.loads(line)
                # Ignore events with missing or wrong uid/type
                if data.get("uid") == uid and data.get("type") == event_type:
                    t_str = data.get("time", "")
                    if t_str:
                        dt = datetime.fromisoformat(t_str.replace("Z", "+00:00"))
                        timestamps.append(dt)
            except Exception:
                continue
    timestamps.sort()
    q = deque()
    max_events = 0

    for t in timestamps:
        # maintain events within [t - window_seconds, t]
        while q and (t - q[0]).total_seconds() > window_seconds:
            q.popleft()
        q.append(t)
        if len(q) > max_events:
            max_events = len(q)
    return max_events

def max_access_after_login(uid, log_path="clue_sample_1GB.jsonl", window_seconds=60):
    """Max number of file_accessed events within window_seconds after login_successful."""
    login_times = []
    access_times = []
    with open(log_path, "r") as f:
        for line in f:
            try:
                data = json.loads(line)
                if data.get("uid") == uid:
                    t_str = data.get("time", "")
                    if not t_str: continue
                    dt = datetime.fromisoformat(t_str.replace("Z", "+00:00"))
                    typ = data.get("type", "")
                    if typ == "login_successful":
                        login_times.append(dt)
                    elif typ == "file_accessed":
                        access_times.append(dt)
            except Exception:
                continue
    login_times.sort()
    access_times.sort()
    max_count = 0
    for login in login_times:
        # Count accesses within window_seconds after each login (access_times sorted)
        idx = 0
        count = 0
        for a in access_times:
            if login < a <= login + timedelta(seconds=window_seconds):
                count += 1
            elif a > login + timedelta(seconds=window_seconds):
                break
        if count > max_count:
            max_count = count
    return max_count

In [16]:
# User configuration: ("uid", "label", multiplier)
users = [
    ("spectacular-copper-cheetah-postman", "Stable", 2),
    ("ready-silver-angelfish-quarryworker", "Instable", 3),
]

event_specs = [
    # (event_type, window_seconds, description, detector_func)
    ("file_written", 60,   "Ransomware (writes in 1 min)", sliding_window_max),
    ("file_deleted", 300,  "Suppression massive (deletes in 5 min)", sliding_window_max),
    ("file_created", 300,  "Depot malveillant (creates in 5 min)", sliding_window_max),
]

print("=========================================")
print("MAXIMA HISTORIQUES ET SEUILS (SLIDING WINDOW)")
print("=========================================\n")

for uid, label, multi in users:
    print(f"{'='*40}\nUtilisateur: {uid}\nProfil: {label} (×{multi})\n{'='*40}")
    # Ransomware
    for event_type, window_sec, desc, func in event_specs:
        m = func(uid, event_type, window_sec)
        print(f"\n{desc}:\n  Max glissant: {m}\n  Seuil (×{multi}): {m * multi}")
    # Sequence case
    seq_max = max_access_after_login(uid, window_seconds=60)
    print(f"\nCompte volé (acces dans 1 min après login):\n  Max glissant: {seq_max}\n  Seuil (×{multi}): {seq_max * multi}\n")

MAXIMA HISTORIQUES ET SEUILS (SLIDING WINDOW)

Utilisateur: spectacular-copper-cheetah-postman
Profil: Stable (×2)

Ransomware (writes in 1 min):
  Max glissant: 3
  Seuil (×2): 6

Suppression massive (deletes in 5 min):
  Max glissant: 1
  Seuil (×2): 2

Depot malveillant (creates in 5 min):
  Max glissant: 3
  Seuil (×2): 6

Compte volé (acces dans 1 min après login):
  Max glissant: 22
  Seuil (×2): 44

Utilisateur: ready-silver-angelfish-quarryworker
Profil: Instable (×3)

Ransomware (writes in 1 min):
  Max glissant: 19
  Seuil (×3): 57

Suppression massive (deletes in 5 min):
  Max glissant: 115
  Seuil (×3): 345

Depot malveillant (creates in 5 min):
  Max glissant: 23
  Seuil (×3): 69

Compte volé (acces dans 1 min après login):
  Max glissant: 116
  Seuil (×3): 348



In [17]:
import numpy as np
import json
from collections import deque
from datetime import datetime, timedelta

def sliding_window_all_counts(uid, event_type, window_seconds, log_path="clue_sample_1GB.jsonl"):
    """Return list of all sliding window counts of event_type for uid."""
    timestamps = []
    with open(log_path, "r") as f:
        for line in f:
            try:
                data = json.loads(line)
                if data.get("uid") == uid and data.get("type") == event_type:
                    t_str = data.get("time", "")
                    if t_str:
                        dt = datetime.fromisoformat(t_str.replace("Z", "+00:00"))
                        timestamps.append(dt)
            except Exception:
                continue
    timestamps.sort()
    q = deque()
    counts = []
    for t in timestamps:
        while q and (t - q[0]).total_seconds() > window_seconds:
            q.popleft()
        q.append(t)
        counts.append(len(q))
    return counts, timestamps

def find_percentile_anomalies(uid, event_type, window_seconds, percentile=95,
                               hard_threshold=None, log_path="clue_sample_1GB.jsonl"):
    counts, timestamps = sliding_window_all_counts(uid, event_type, window_seconds, log_path)
    if not counts:
        print(f"  No data for {event_type}")
        return

    p_threshold = np.percentile(counts, percentile)
    p_threshold = max(1, int(np.ceil(p_threshold)))

    threshold = min(p_threshold, hard_threshold) if hard_threshold is not None else p_threshold
    print(f"  Percentile {percentile}% threshold: {p_threshold} → using: {threshold}")

    start = 0
    in_burst = False
    peak_count = 0
    peak_start = None
    peak_end = None
    burst_wall_end = None

    for end, t in enumerate(timestamps):
        window_start_time = t - timedelta(seconds=window_seconds)
        while start <= end and timestamps[start] < window_start_time:
            start += 1

        full_count = end - start + 1

        if full_count >= threshold:
            if not in_burst:
                in_burst = True
                peak_count = full_count
                peak_start = timestamps[start]
                peak_end = t
                burst_wall_end = t + timedelta(seconds=window_seconds)
            else:
                burst_wall_end = t + timedelta(seconds=window_seconds)
                if full_count > peak_count:
                    peak_count = full_count
                    peak_start = timestamps[start]
                    peak_end = t
        else:
            if in_burst and t > burst_wall_end:
                print(f"    🚨 {event_type}: {peak_count} events from {peak_start} to {peak_end}")
                in_burst = False
                peak_count = 0
                burst_wall_end = None

    if in_burst:
        print(f"    🚨 {event_type}: {peak_count} events from {peak_start} to {peak_end}")

# --- CONFIG ---
users = [
    ("spectacular-copper-cheetah-postman", "Stable"),
    ("ready-silver-angelfish-quarryworker", "Instable")
]
event_specs = [
    ("file_written", 60,  "Ransomware (writes in 1 min)",          None),
    ("file_deleted", 300, "Suppression massive (deletes in 5 min)", 10),
    ("file_created", 300, "Depot malveillant (creates in 5 min)",   None),
]
percentile = 95

# --- MAIN ---
print(f"==== PERCENTILE-BASED SLIDING WINDOW ANOMALY DETECTION (percentile={percentile}) ====")
for uid, label in users:
    print(f"\n{'='*25}\nUser: {uid} | Profile: {label}\n{'='*25}")
    for event_type, window_sec, desc, hard_thresh in event_specs:
        print(f"-- {desc}:")
        find_percentile_anomalies(uid, event_type, window_sec, percentile=percentile,
                                   hard_threshold=hard_thresh)

==== PERCENTILE-BASED SLIDING WINDOW ANOMALY DETECTION (percentile=95) ====

User: spectacular-copper-cheetah-postman | Profile: Stable
-- Ransomware (writes in 1 min):
  Percentile 95% threshold: 3 → using: 3
    🚨 file_written: 3 events from 2017-08-08 07:46:16+00:00 to 2017-08-08 07:46:41+00:00
-- Suppression massive (deletes in 5 min):
  Percentile 95% threshold: 1 → using: 1
    🚨 file_deleted: 1 events from 2017-08-08 07:46:24+00:00 to 2017-08-08 07:46:24+00:00
-- Depot malveillant (creates in 5 min):
  Percentile 95% threshold: 3 → using: 3
    🚨 file_created: 3 events from 2017-08-08 07:46:16+00:00 to 2017-08-08 07:46:41+00:00

User: ready-silver-angelfish-quarryworker | Profile: Instable
-- Ransomware (writes in 1 min):
  Percentile 95% threshold: 9 → using: 9
    🚨 file_written: 9 events from 2017-07-29 06:02:05+00:00 to 2017-07-29 06:02:41+00:00
    🚨 file_written: 19 events from 2017-08-19 09:10:44+00:00 to 2017-08-19 09:11:11+00:00
    🚨 file_written: 10 events from 2017-0